**Install Required Libraries**


In [ ]:
!pip -q install lxml pandas openpyxl rdflib scikit-learn

**Import Libraries and Configure Environment**


In [ ]:
import os
import re
import json
import zipfile
import random
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

from collections import Counter
from google.colab import drive
from sklearn.model_selection import train_test_split

from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

**Mount Google Drive**


In [ ]:
drive.mount('/content/drive')

**Load and Extract DrugBank Source Files**


In [ ]:
# Define input ZIP path, extraction folder, and output folder for the DrugBank pipeline
ZIP_PATH = "/content/drive/MyDrive/Depixen/drugbank_all_full_database.xml.zip"
EXTRACT_DIR = "/content/drugbank_extracted_final"
OUTPUT_DIR = "/content/drive/MyDrive/Depixen/drugbank_final_clean_pipeline"

# Create extraction and output directories if they do not already exist
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check whether the DrugBank ZIP file exists before extraction
print("ZIP exists:", os.path.exists(ZIP_PATH))

# Extract all contents of the ZIP file into the working extraction directory
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# Display the extracted files to confirm successful extraction
print("Extracted files:")
for f in os.listdir(EXTRACT_DIR):
    print("-", f)

# Find XML files from the extracted contents
xml_files = [f for f in os.listdir(EXTRACT_DIR) if f.endswith(".xml")]

# Stop execution if no XML file is found after extraction
if not xml_files:
    raise FileNotFoundError("No XML file found after extraction.")

# Build the full path of the extracted XML file for downstream parsing
XML_PATH = os.path.join(EXTRACT_DIR, xml_files[0])
print("\nXML_PATH:", XML_PATH)

**Parse DrugBank XML and Detect Namespace**


In [ ]:
# Parse the extracted DrugBank XML file and load its root element
tree = ET.parse(XML_PATH)
root = tree.getroot()

# Print the root tag to inspect the XML document structure
print("Root tag:", root.tag)

# Extract the XML namespace from the root tag if present
match = re.match(r"\{(.*)\}", root.tag)
ns_uri = match.group(1) if match else ""
ns = {"db": ns_uri} if ns_uri else {}

# Print the namespace URI for use in namespace-aware XML queries
print("Namespace URI:", ns_uri)

# Find all drug elements in the XML and count total DrugBank records
drug_elements = root.findall("db:drug", ns) if ns else root.findall("drug")
print("Total drug records found:", len(drug_elements))

**Explore XML Record Structure**


In [ ]:
# Define a helper function to remove the XML namespace from element tags
def strip_namespace(tag):
    return tag.split("}")[-1] if "}" in tag else tag

# Select the first drug record from the XML to inspect its internal structure
first_drug = root[0]

# Print the cleaned tag name of the first drug element
print("First drug tag:", strip_namespace(first_drug.tag))
print("\nDirect child tags in first drug record:\n")

# Loop through the first drug record and print all direct child tag names
for child in first_drug:
    print(strip_namespace(child.tag))

**Analyse Top-Level XML Fields**


In [ ]:
# Count how many times each top-level XML field appears across all DrugBank drug records
top_level_counter = Counter()

# Iterate through every drug record and count each direct child tag as a top-level column
for drug in root:
    for child in drug:
        top_level_counter[strip_namespace(child.tag)] += 1

# Convert the top-level field counts into a DataFrame and sort by frequency
top_level_df = pd.DataFrame(
    [{"column_name": k, "occurrences": v} for k, v in top_level_counter.items()]
).sort_values(by="occurrences", ascending=False).reset_index(drop=True)

# Print the total number of unique top-level columns identified in the XML
print("Total unique top-level columns found:", len(top_level_df))
display(top_level_df.head(55))

**Infer XML Column Data Types**


In [ ]:
# Check whether an XML element contains meaningful text, attributes, or nested text content
def has_value(element):
    if element is None:
        return False
    if (element.text or "").strip():
        return True
    if element.attrib:
        return True
    for child in element.iter():
        if child is not element and (child.text or "").strip():
            return True
    return False

# Detect the likely datatype of a selected DrugBank XML column using sampled values
def detect_datatype_for_column(root, column_name, sample_limit=200):
    values = []

    # Collect sample values from the requested column across drug records
    for drug in root:
        for child in drug:
            if strip_namespace(child.tag) == column_name and has_value(child):
                direct_text = (child.text or "").strip()
                child_count = len(list(child))

                # Mark the column as nested or mixed if it contains child elements
                if child_count > 0:
                    return "nested_or_mixed"

                # Store direct text values for datatype inspection
                if direct_text:
                    values.append(direct_text)

                # Stop sampling once the defined sample limit is reached
                if len(values) >= sample_limit:
                    break

        # Stop outer loop once enough sample values are collected
        if len(values) >= sample_limit:
            break

    # Mark the column as empty if no usable values are found
    if not values:
        return "empty"

    numeric_count = 0
    for v in values:
        try:
            float(v)
            numeric_count += 1
        except:
            pass

    # Mark the column as numeric if all sampled values can be converted to numbers
    if numeric_count == len(values):
        return "numeric"

    # Mark the column as text if values are non-numeric plain text
    return "text"

**Generate XML Completeness Report**


In [ ]:
# Calculate the total number of drug records available in the DrugBank XML
total_records = len(root)
print("Total records:", total_records)

# Collect all unique top-level column names present across all drug records
all_columns = set()
for drug in root:
    for child in drug:
        all_columns.add(strip_namespace(child.tag))
all_columns = sorted(all_columns)

# Print the number of unique top-level columns discovered in the XML structure
print("Total unique top-level columns:", len(all_columns))

# Create an empty list to store completeness and datatype summary for each column
summary_rows = []

# Iterate through each top-level column and calculate completeness statistics
for col in all_columns:
    non_null_count = 0

    # Count how many drug records contain this column with a meaningful value
    for drug in root:
        found_with_value = False
        for child in drug:
            if strip_namespace(child.tag) == col and has_value(child):
                found_with_value = True
                break
        if found_with_value:
            non_null_count += 1

    # Compute null counts, completeness percentages, and inferred datatype for the column
    null_count = total_records - non_null_count
    non_null_percentage = round((non_null_count / total_records) * 100, 2)
    null_percentage = round((null_count / total_records) * 100, 2)
    data_type = detect_datatype_for_column(root, col)

    # Store the summary statistics for the current column
    summary_rows.append({
        "column_name": col,
        "total_records": total_records,
        "non_null_count": non_null_count,
        "null_count": null_count,
        "non_null_percentage": non_null_percentage,
        "null_percentage": null_percentage,
        "data_type": data_type
    })

# Convert the column summary list into a DataFrame and sort it by coverage and column name
column_summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["non_null_count", "column_name"],
    ascending=[False, True]
).reset_index(drop=True)

# Expand pandas display settings to show the full summary table clearly
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

# Display the final top-level column completeness and datatype summary table
display(column_summary_df)

**Select Final Fields for Extraction**


In [ ]:
FINAL_SELECTED_COLUMNS = [
    "drugbank-id",
    "name",
    "groups",
    "synonyms",
    "state",
    "unii",
    "cas-number",
    "average-mass",
    "monoisotopic-mass",
    "description",
    "indication",
    "mechanism-of-action",
]

print("Total selected fields:", len(FINAL_SELECTED_COLUMNS))
print(FINAL_SELECTED_COLUMNS)

Define XML Text Extraction Utilities


In [ ]:
# Clean raw text by trimming spaces and normalizing repeated whitespace
def clean_text(text):
    if text is None:
        return None
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text if text else None

# Extract and clean text from a single XML child element using the given path
def get_text(elem, path):
    child = elem.find(path, ns) if ns else elem.find(path)
    if child is not None and child.text:
        return clean_text(child.text)
    return None

# Extract, clean, deduplicate, and join text values from multiple XML elements
def get_all_texts(elem, path):
    nodes = elem.findall(path, ns) if ns else elem.findall(path)
    values = []

    # Collect all non-empty text values from matching XML nodes
    for node in nodes:
        if node is not None and node.text:
            val = clean_text(node.text)
            if val:
                values.append(val)

    # Remove duplicate values while preserving the original order
    seen = set()
    ordered = []
    for v in values:
        if v not in seen:
            seen.add(v)
            ordered.append(v)

    # Return all unique values as a single pipe-separated string
    return "|".join(ordered) if ordered else None

# Extract all drug group values from the DrugBank XML record
def get_group_texts(drug):
    return get_all_texts(drug, "db:groups/db:group" if ns else "groups/group")

# Extract all synonym values from the DrugBank XML record
def get_synonyms_texts(drug):
    return get_all_texts(drug, "db:synonyms/db:synonym" if ns else "synonyms/synonym")

# Extract all DrugBank IDs linked to the drug record
def get_drugbank_ids(drug):
    return get_all_texts(drug, "db:drugbank-id" if ns else "drugbank-id")

# Extract the drug description text
def get_description(drug):
    return get_text(drug, "db:description" if ns else "description")

# Extract the drug indication text
def get_indication(drug):
    return get_text(drug, "db:indication" if ns else "indication")

# Extract the drug mechanism of action text
def get_mechanism(drug):
    return get_text(drug, "db:mechanism-of-action" if ns else "mechanism-of-action")

# Extract the physical state of the drug
def get_state(drug):
    return get_text(drug, "db:state" if ns else "state")

# Extract the primary drug name
def get_name(drug):
    return get_text(drug, "db:name" if ns else "name")

# Extract the UNII code of the drug
def get_unii(drug):
    return get_text(drug, "db:unii" if ns else "unii")

# Extract the CAS number of the drug
def get_cas(drug):
    return get_text(drug, "db:cas-number" if ns else "cas-number")

# Extract the average molecular mass of the drug
def get_average_mass(drug):
    return get_text(drug, "db:average-mass" if ns else "average-mass")

# Extract the monoisotopic molecular mass of the drug
def get_monoisotopic_mass(drug):
    return get_text(drug, "db:monoisotopic-mass" if ns else "monoisotopic-mass")

**Extract DrugBank Records into Tabular Format**


In [ ]:
# Create an empty list to store extracted DrugBank records as structured rows
rows = []

# Extract the selected 12 fields from each DrugBank drug record
for drug in drug_elements:
    row = {
        "drugbank-id": get_drugbank_ids(drug),
        "name": get_name(drug),
        "groups": get_group_texts(drug),
        "synonyms": get_synonyms_texts(drug),
        "state": get_state(drug),
        "description": get_description(drug),
        "indication": get_indication(drug),
        "mechanism-of-action": get_mechanism(drug),
        "unii": get_unii(drug),
        "cas-number": get_cas(drug),
        "average-mass": get_average_mass(drug),
        "monoisotopic-mass": get_monoisotopic_mass(drug),
    }
    rows.append(row)

# Convert the extracted drug records into a pandas DataFrame
raw_df = pd.DataFrame(rows)

# Print the shape of the raw extracted dataset and preview the first records
print("Raw dataframe shape:", raw_df.shape)
display(raw_df.head(3))

**Audit Raw Extracted Dataset**


In [ ]:
# Create a summary list to store completeness statistics for each extracted raw dataset column
summary_rows = []
total_records = len(raw_df)

# Calculate non-null count, null count, and percentage coverage for each raw DataFrame column
for col in raw_df.columns:
    non_null = raw_df[col].notna().sum()
    null_count = raw_df[col].isna().sum()
    non_null_pct = (non_null / total_records) * 100 if total_records else 0
    null_pct = (null_count / total_records) * 100 if total_records else 0

    # Store column-level data quality summary for the raw extracted dataset
    summary_rows.append({
        "column": col,
        "dtype": str(raw_df[col].dtype),
        "non_null_count": non_null,
        "null_count": null_count,
        "non_null_pct": round(non_null_pct, 2),
        "null_pct": round(null_pct, 2),
    })

# Convert the raw column summary into a DataFrame and sort by completeness and column name
raw_column_summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["non_null_count", "column"],
    ascending=[False, True]
).reset_index(drop=True)

# Display the raw dataset column completeness summary table
display(raw_column_summary_df)

**Save Raw Dataset Outputs**


In [ ]:
raw_df.to_csv(os.path.join(OUTPUT_DIR, "drugbank_raw_selected_columns.csv"), index=False)
raw_column_summary_df.to_csv(os.path.join(OUTPUT_DIR, "drugbank_raw_column_summary.csv"), index=False)

print("Saved:")
print("-", os.path.join(OUTPUT_DIR, "drugbank_raw_selected_columns.csv"))
print("-", os.path.join(OUTPUT_DIR, "drugbank_raw_column_summary.csv"))

**Define Dataset Cleaning Utilities**


In [ ]:
# Normalize a cell value by cleaning whitespace and converting empty values to None
def normalize_cell(x):
    if pd.isna(x) or x is None:
        return None
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x if x else None

# Split a pipe-separated string into a clean list of individual values
def split_pipe_values(value):
    if pd.isna(value) or value is None:
        return []
    return [v.strip() for v in str(value).split("|") if v.strip()]

**Define ID and Text Standardisation Rules**


In [ ]:
# Extract the main DrugBank ID from multiple IDs by selecting the first valid DB-formatted identifier
def extract_primary_dbid(x):
    if pd.isna(x) or x is None:
        return None

    ids = [i.strip() for i in str(x).split("|") if i.strip()]

    for i in ids:
        if re.match(r"^DB\d+$", i):
            return i

    return None

# Keep only the first few unique synonyms to control synonym length in the final dataset
def limit_synonyms(x, max_syn=2):
    if pd.isna(x) or x is None:
        return None

    syns = [s.strip() for s in str(x).split("|") if s.strip()]

    seen = set()
    cleaned = []
    for s in syns:
        key = s.lower()
        if key not in seen:
            seen.add(key)
            cleaned.append(s)

    cleaned = cleaned[:max_syn]
    return "|".join(cleaned) if cleaned else None

# Clean free-text fields by removing HTML, bracketed content, URLs, and extra whitespace
def clean_text_basic(text):
    if pd.isna(text) or text is None:
        return None

    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\[[^\]]+\]", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None

# Extract a short first sentence or chunk from long text fields for concise and cleaner output
def extract_first_short_chunk(text, max_words=30):
    if pd.isna(text) or text is None:
        return None

    text = clean_text_basic(text)
    if not text:
        return None

    split_patterns = [
        r"\.\s+",
        r"\?\s+",
        r"!\s+",
        r";\s+",
        r":\s+",
    ]

    first = text

    # Split the text at the first meaningful sentence boundary and keep the first chunk
    for pattern in split_patterns:
        parts = re.split(pattern, first, maxsplit=1)
        if parts and parts[0].strip() and len(parts[0].split()) >= 4:
            first = parts[0].strip()
            break

    # Limit the extracted chunk to a maximum word count for consistency
    words = first.split()
    if len(words) > max_words:
        first = " ".join(words[:max_words]).rstrip(",;:")

    # Ensure the final cleaned text ends with a full stop
    if not first.endswith("."):
        first += "."

    return first

**Clean and Standardise the 12-Column Dataset**


In [ ]:
# Select only the final 12 approved columns from the raw DrugBank dataset for cleaning
clean_12_df = raw_df[FINAL_SELECTED_COLUMNS].copy()

# Apply general cell normalization to standardize whitespace and empty values across all selected columns
for col in clean_12_df.columns:
    clean_12_df[col] = clean_12_df[col].apply(normalize_cell)

# Normalize DrugBank IDs by keeping only the primary valid DB identifier
clean_12_df["drugbank-id"] = clean_12_df["drugbank-id"].apply(extract_primary_dbid)

# Limit synonyms to the first two unique values to keep the dataset concise and controlled
clean_12_df["synonyms"] = clean_12_df["synonyms"].apply(lambda x: limit_synonyms(x, max_syn=2))

# Shorten long description text into a compact first sentence or chunk
clean_12_df["description"] = clean_12_df["description"].apply(lambda x: extract_first_short_chunk(x, max_words=30))

# Shorten long indication text into a compact first sentence or chunk
clean_12_df["indication"] = clean_12_df["indication"].apply(lambda x: extract_first_short_chunk(x, max_words=25))

# Shorten long mechanism-of-action text into a compact first sentence or chunk
clean_12_df["mechanism-of-action"] = clean_12_df["mechanism-of-action"].apply(lambda x: extract_first_short_chunk(x, max_words=30))

# Reapply normalization after all field-specific cleaning operations
for col in clean_12_df.columns:
    clean_12_df[col] = clean_12_df[col].apply(normalize_cell)

# Print the cleaned dataset shape and preview the first records after applying all cleaning logic
print("Shape after applying logic:", clean_12_df.shape)
display(clean_12_df.head(5))

**Remove Incomplete Records**

In [ ]:
# Remove rows that contain null values in any of the final selected columns to ensure complete records
clean_12_df = clean_12_df.dropna(subset=FINAL_SELECTED_COLUMNS).reset_index(drop=True)

# Display the dataset shape and preview after removing incomplete rows
print("Shape after removing all null rows:", clean_12_df.shape)
display(clean_12_df.head(5))

In [ ]:
# Convert all categorical and text-based DrugBank columns to string type for consistency
clean_12_df["drugbank-id"] = clean_12_df["drugbank-id"].astype("string")
clean_12_df["name"] = clean_12_df["name"].astype("string")
clean_12_df["groups"] = clean_12_df["groups"].astype("string")
clean_12_df["synonyms"] = clean_12_df["synonyms"].astype("string")
clean_12_df["state"] = clean_12_df["state"].astype("string")
clean_12_df["unii"] = clean_12_df["unii"].astype("string")
clean_12_df["cas-number"] = clean_12_df["cas-number"].astype("string")
clean_12_df["description"] = clean_12_df["description"].astype("string")
clean_12_df["indication"] = clean_12_df["indication"].astype("string")
clean_12_df["mechanism-of-action"] = clean_12_df["mechanism-of-action"].astype("string")

# Convert molecular mass columns to numeric type and coerce invalid values to NaN
clean_12_df["average-mass"] = pd.to_numeric(clean_12_df["average-mass"], errors="coerce")
clean_12_df["monoisotopic-mass"] = pd.to_numeric(clean_12_df["monoisotopic-mass"], errors="coerce")

# Remove rows where either molecular mass value is missing after numeric conversion
clean_12_df = clean_12_df.dropna(subset=["average-mass", "monoisotopic-mass"]).reset_index(drop=True)

# Print final data types and preview the cleaned dataset after type enforcement
print(clean_12_df.dtypes)
display(clean_12_df.head(5))

**Enforce Final Column Data Types**


In [ ]:
# Create a final summary table to measure completeness and datatype of each cleaned dataset column
summary_rows = []
total_records = len(clean_12_df)

# Calculate non-null count, null count, percentage coverage, and datatype for each final column
for col in clean_12_df.columns:
    non_null_count = clean_12_df[col].notna().sum()
    null_count = clean_12_df[col].isna().sum()
    non_null_percentage = round((non_null_count / total_records) * 100, 2) if total_records else 0
    null_percentage = round((null_count / total_records) * 100, 2) if total_records else 0

    # Store the final completeness and datatype summary for the cleaned dataset
    summary_rows.append({
        "column_name": col,
        "total_records": total_records,
        "non_null_count": non_null_count,
        "null_count": null_count,
        "non_null_percentage": non_null_percentage,
        "null_percentage": null_percentage,
        "data_type": str(clean_12_df[col].dtype)
    })

# Convert the final column summary into a DataFrame for inspection
column_summary_final_df = pd.DataFrame(summary_rows)

# Display the final cleaned dataset column summary table
display(column_summary_final_df)

**Audit Final Clean Dataset**


In [ ]:
clean_12_df.to_csv(os.path.join(OUTPUT_DIR, "drugbank_clean_12cols.csv"), index=False)
column_summary_final_df.to_csv(os.path.join(OUTPUT_DIR, "drugbank_clean_12cols_summary.csv"), index=False)

print("Saved:")
print("-", os.path.join(OUTPUT_DIR, "drugbank_clean_12cols.csv"))
print("-", os.path.join(OUTPUT_DIR, "drugbank_clean_12cols_summary.csv"))


**Define RDF Ontology for DrugBank KG**

In [ ]:
# Define the base namespace and create an RDF graph for the DrugBank ontology
drugkg = Namespace("http://example.org/drugkg/")
g_ontology = Graph()

# Bind common RDF namespaces for readable ontology serialization
g_ontology.bind("drugkg", drugkg)
g_ontology.bind("rdf", RDF)
g_ontology.bind("rdfs", RDFS)
g_ontology.bind("owl", OWL)
g_ontology.bind("xsd", XSD)

# Define the main Drug class in the ontology
g_ontology.add((drugkg.Drug, RDF.type, OWL.Class))
g_ontology.add((drugkg.Drug, RDFS.label, Literal("Drug")))
g_ontology.add((drugkg.Drug, RDFS.comment, Literal("A DrugBank drug entity.")))

# Define the ontology datatype properties and their expected XML schema data types
ontology_properties = {
    "hasDrugBankID": XSD.string,
    "hasName": XSD.string,
    "hasGroup": XSD.string,
    "hasSynonym": XSD.string,
    "hasState": XSD.string,
    "hasUNII": XSD.string,
    "hasCASNumber": XSD.string,
    "hasAverageMass": XSD.float,
    "hasMonoisotopicMass": XSD.float,
    "hasDescription": XSD.string,
    "hasIndication": XSD.string,
    "hasMechanismOfAction": XSD.string,
}

# Add each datatype property to the ontology with its domain, range, and label
for prop, rng in ontology_properties.items():
    g_ontology.add((drugkg[prop], RDF.type, OWL.DatatypeProperty))
    g_ontology.add((drugkg[prop], RDFS.domain, drugkg.Drug))
    g_ontology.add((drugkg[prop], RDFS.range, rng))
    g_ontology.add((drugkg[prop], RDFS.label, Literal(prop)))

**Inspect Ontology Triples**


In [ ]:
print("Ontology triple count:", len(g_ontology))

for triple in list(g_ontology)[:50]:
    print(triple)

**Preview Ontology Turtle Content**


In [ ]:
ontology_ttl = g_ontology.serialize(format="turtle")
print(ontology_ttl)

**Save RDF Ontology File**


In [ ]:
ontology_ttl_path = os.path.join(OUTPUT_DIR, "drugbank_ontology.owl.ttl")
g_ontology.serialize(destination=ontology_ttl_path, format="turtle")

print("Saved ontology:")
print("-", ontology_ttl_path)

**Generate RDF Triples from Clean Data**


In [ ]:
# Create RDF-style subject-predicate-object triples from the final cleaned DrugBank dataset
triples_data_v2 = []

# Convert each cleaned drug record into multiple structured triples for knowledge graph construction
for _, row in clean_12_df.iterrows():
    subject = str(row["drugbank-id"]).strip()

    # Skip records that do not have a valid subject identifier
    if not subject:
        continue

    # Add the core identity and type triples for the current drug entity
    triples_data_v2.append({"subject": subject, "predicate": "rdf:type", "object": "Drug"})
    triples_data_v2.append({"subject": subject, "predicate": "hasName", "object": str(row["name"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasDrugBankID", "object": subject})

    # Add one triple per synonym value for the current drug
    for syn in split_pipe_values(row["synonyms"]):
        triples_data_v2.append({"subject": subject, "predicate": "hasSynonym", "object": syn})

    # Add one triple per group value for the current drug
    for grp in split_pipe_values(row["groups"]):
        triples_data_v2.append({"subject": subject, "predicate": "hasGroup", "object": grp})

    # Add the remaining attribute triples for the current drug
    triples_data_v2.append({"subject": subject, "predicate": "hasState", "object": str(row["state"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasUNII", "object": str(row["unii"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasCASNumber", "object": str(row["cas-number"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasAverageMass", "object": str(row["average-mass"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasMonoisotopicMass", "object": str(row["monoisotopic-mass"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasDescription", "object": str(row["description"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasIndication", "object": str(row["indication"]).strip()})
    triples_data_v2.append({"subject": subject, "predicate": "hasMechanismOfAction", "object": str(row["mechanism-of-action"]).strip()})

# Convert the generated triples into a DataFrame for inspection and downstream use
triples_df_v2 = pd.DataFrame(triples_data_v2)

# Print the triple dataset shape and preview the first generated triples
print("Triples dataframe shape:", triples_df_v2.shape)
display(triples_df_v2.head(20))

**Analyse RDF Triple Distribution**


In [ ]:
triple_counts_v2 = triples_df_v2.groupby("subject").size().reset_index(name="triple_count")

print("Average triples per drug:", triple_counts_v2["triple_count"].mean())
print("Max triples per drug:", triple_counts_v2["triple_count"].max())
print("Min triples per drug:", triple_counts_v2["triple_count"].min())

display(triple_counts_v2.head())

**Generate KG-to-Text Instruction Dataset**


In [ ]:
# Import required libraries for text cleaning, random template selection, and dataset preparation
import pandas as pd
import re
import random

# Set a fixed random seed to make template and instruction selection reproducible
random.seed(42)

# Safely convert missing values to empty strings and strip whitespace from valid values
def safe(v):
    if pd.isna(v):
        return ""
    return str(v).strip()


# Clean sentence-level text by removing bracketed content and normalizing whitespace and punctuation spacing
def clean_sentence_text(text):
    text = safe(text)
    if not text:
        return ""

    text = re.sub(r"\[[^\]]+\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    return text


# Clean field text and repair common grammatical errors produced during text generation preparation
def clean_field_text(text):
    text = clean_sentence_text(text)
    if not text:
        return ""

    repairs = [
        (r"\bit is indicated for investigated for\b", "it is indicated for"),
        (r"\bit is indicated for used for\b", "it is used for"),
        (r"\bit is indicated for indicated for\b", "it is indicated for"),
        (r"\bits mechanism of action involves is\b", "its mechanism of action involves"),
        (r"\bit works through is\b", "it works through"),
        (r"\bit works by is\b", "it works by"),
        (r"\ba appetite\b", "an appetite"),
        (r"\ba anti\b", "an anti"),
        (r"\ba approved\b", "an approved"),
        (r"\ba investigational\b", "an investigational"),
        (r"\ba experimental\b", "an experimental"),
        (r"\ba oral\b", "an oral"),
    ]

    for pattern, replacement in repairs:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text.strip()


# Convert the first character of text to lowercase for sentence continuation use
def lower_first_char(text):
    text = clean_field_text(text)
    if not text:
        return ""
    return text[0].lower() + text[1:] if len(text) > 1 else text.lower()


# Convert the first character of text to uppercase for sentence opening use
def upper_first_char(text):
    text = clean_field_text(text)
    if not text:
        return ""
    return text[0].upper() + text[1:] if len(text) > 1 else text.upper()


# Ensure that a text fragment ends with valid sentence-ending punctuation
def ensure_period(text):
    text = clean_field_text(text)
    if not text:
        return ""
    if text.endswith((".", "!", "?")):
        return text
    return text + "."


# Remove trailing sentence-ending punctuation from a text fragment
def strip_trailing_period(text):
    text = clean_field_text(text)
    if not text:
        return ""
    return text.rstrip(".!?").strip()


# Check whether a text begins with a vowel sound for article selection
def starts_with_vowel_sound(text):
    text = safe(text).lower()
    if not text:
        return False
    return text[0] in "aeiou"


# Choose the correct indefinite article based on the starting sound of the text
def choose_article(text):
    return "an" if starts_with_vowel_sound(text) else "a"


# Split, clean, deduplicate, and normalize pipe-separated values
def normalize_pipe_values(text):
    text = safe(text)
    if not text:
        return []

    values = [x.strip() for x in text.split("|") if x.strip()]
    values = [clean_sentence_text(x) for x in values if clean_sentence_text(x)]

    seen = set()
    deduped = []

    for v in values:
        key = v.lower()
        if key not in seen:
            deduped.append(v)
            seen.add(key)

    return deduped


# Join multiple values into a natural language phrase using commas and 'and'
def join_natural(values):
    cleaned = []
    seen = set()

    for v in values:
        v_clean = clean_sentence_text(v)
        if not v_clean:
            continue

        key = v_clean.lower()
        if key not in seen:
            cleaned.append(v_clean)
            seen.add(key)

    if not cleaned:
        return ""
    if len(cleaned) == 1:
        return cleaned[0]
    if len(cleaned) == 2:
        return f"{cleaned[0]} and {cleaned[1]}"
    return ", ".join(cleaned[:-1]) + f", and {cleaned[-1]}"


# Normalize drug group values, prioritize important group types, and keep only the top groups
def normalize_group_values(group_text, max_groups=2):
    values = normalize_pipe_values(group_text)
    if not values:
        return []

    priority = {
        "approved": 1,
        "investigational": 2,
        "experimental": 3,
        "withdrawn": 4,
        "illicit": 5,
        "nutraceutical": 6,
        "vet_approved": 7
    }

    values = sorted(values, key=lambda x: priority.get(x.lower(), 999))
    return values[:max_groups]


# Normalize synonym values, remove duplicates and the main drug name, and keep only short useful synonyms
def normalize_synonyms_text(syn_text, main_name="", max_synonyms=2):
    values = normalize_pipe_values(syn_text)
    if not values:
        return ""

    cleaned = []
    seen = set()

    for v in values:
        v_clean = clean_sentence_text(v)
        if not v_clean:
            continue

        if main_name and v_clean.lower() == main_name.lower():
            continue

        if len(v_clean) > 120:
            continue

        key = v_clean.lower()
        if key not in seen:
            cleaned.append(v_clean)
            seen.add(key)

    cleaned = cleaned[:max_synonyms]
    return join_natural(cleaned)


# Convert grouped triples into ordered multiline input text for KG-to-text training
def triples_to_input_text(group_df):
    preferred_order = {
        "rdf:type": 1,
        "hasName": 2,
        "hasDrugBankID": 3,
        "hasSynonym": 4,
        "hasGroup": 5,
        "hasState": 6,
        "hasDescription": 7,
        "hasIndication": 8,
        "hasMechanismOfAction": 9,
        "hasUNII": 10,
        "hasCASNumber": 11,
        "hasAverageMass": 12,
        "hasMonoisotopicMass": 13
    }

    ordered_df = group_df.copy()
    ordered_df["predicate"] = ordered_df["predicate"].astype(str).str.strip()
    ordered_df["subject"] = ordered_df["subject"].astype(str).str.strip()
    ordered_df["object"] = ordered_df["object"].astype(str).str.strip()
    ordered_df["object_clean"] = ordered_df["object"].apply(clean_sentence_text)
    ordered_df["sort_order"] = ordered_df["predicate"].map(lambda x: preferred_order.get(x, 999))

    # Sort triples in a consistent logical order before converting them to training input text
    ordered_df = ordered_df.sort_values(
        by=["sort_order", "predicate", "object_clean"],
        ascending=[True, True, True]
    ).drop(columns=["sort_order"])

    lines = []

    # Convert each triple into a structured line in subject-predicate-object format
    for _, r in ordered_df.iterrows():
        subj = safe(r["subject"])
        pred = safe(r["predicate"])
        obj = safe(r["object_clean"])

        if subj and pred and obj:
            lines.append(f"({subj}, {pred}, {obj})")

    return "\n".join(lines)


# Clean and finalize the description field as a proper sentence
def clean_description_text(desc):
    desc = clean_field_text(desc)
    if not desc:
        return ""

    desc = upper_first_char(desc)
    desc = ensure_period(desc)
    return desc


# Build a fluent indication sentence from the indication field
def build_indication_sentence(indication):
    indication = clean_field_text(indication)
    if not indication:
        return ""

    indication_text = lower_first_char(indication)
    indication_text = strip_trailing_period(indication_text)

    if indication_text.startswith("for "):
        return f"It is indicated {indication_text}."
    if indication_text.startswith(("used for ", "used in ", "intended for ", "approved for ", "investigated for ")):
        return f"It is {indication_text}."
    return f"It is indicated for {indication_text}."


# Build a fluent mechanism-of-action sentence from the mechanism field
def build_mechanism_sentence(moa):
    moa = clean_field_text(moa)
    if not moa:
        return ""

    moa_text = lower_first_char(moa)
    moa_text = strip_trailing_period(moa_text)

    if moa_text.startswith(("by ", "through ", "via ")):
        return f"It acts {moa_text}."
    if moa_text.startswith(("inhibits ", "binds ", "blocks ", "activates ", "modulates ", "stimulates ", "suppresses ", "reduces ", "increases ")):
        return f"It works by {moa_text}."
    if moa_text.startswith(("the inhibition", "the activation", "the modulation", "the suppression", "the reduction", "the increase")):
        return f"Its mechanism of action involves {moa_text}."
    if moa_text.startswith("the "):
        return f"Its mechanism of action involves {moa_text}."
    return f"It works through {moa_text}."


# Build the opening sentence describing the drug identity, DrugBank ID, and drug group
def build_opening_sentence(name, groups, drugbank_id):
    groups = [g for g in groups if g]

    if name and drugbank_id:
        opening = f"{name} is a drug recorded in DrugBank under the identifier {drugbank_id}"
    elif name:
        opening = f"{name} is a drug"
    elif drugbank_id:
        opening = f"This drug is recorded in DrugBank under the identifier {drugbank_id}"
    else:
        opening = "This is a drug"

    if len(groups) == 1:
        article = choose_article(groups[0])
        opening += f" and is {article} {groups[0]} drug"
    elif len(groups) == 2:
        opening += f" and belongs to the groups {groups[0]} and {groups[1]}"
    elif len(groups) > 2:
        opening += f" and belongs to the groups {join_natural(groups)}"

    return opening + "."


# Build a synonym sentence using one or more selected synonyms
def build_synonym_sentence(synonyms, raw_synonyms):
    if not synonyms:
        return ""

    raw_synonyms = safe(raw_synonyms)
    syn_values = normalize_pipe_values(raw_synonyms)

    if len(syn_values) <= 1:
        return f"It is also known as {synonyms}."
    return f"It is also known by the following synonyms: {synonyms}."


# Build a sentence describing physical state and molecular mass attributes
def build_physical_sentence(state, avg_mass, mono_mass):
    state = clean_field_text(state)
    avg_mass = clean_field_text(avg_mass)
    mono_mass = clean_field_text(mono_mass)

    phrases = []

    if state:
        phrases.append(f"it is present in {state} form")
    if avg_mass:
        phrases.append(f"it has an average mass of {avg_mass}")
    if mono_mass:
        phrases.append(f"it has a monoisotopic mass of {mono_mass}")

    if not phrases:
        return ""

    if len(phrases) == 1:
        return upper_first_char(phrases[0]) + "."
    if len(phrases) == 2:
        return upper_first_char(f"{phrases[0]}, and {phrases[1]}.")
    return upper_first_char(f"{phrases[0]}, {phrases[1]}, and {phrases[2]}.")


# Build a registry sentence using UNII and CAS identifiers
def build_registry_sentence(unii, cas):
    unii = clean_field_text(unii)
    cas = clean_field_text(cas)

    if unii and cas:
        return f"Its UNII is {unii}, and its CAS number is {cas}."
    elif unii:
        return f"Its UNII is {unii}."
    elif cas:
        return f"Its CAS number is {cas}."
    return ""


# Remove duplicate or empty sentence parts before final text assembly
def deduplicate_sentences(parts):
    seen = set()
    cleaned_parts = []

    for part in parts:
        part = clean_field_text(part)
        if not part:
            continue

        key = part.lower()
        if key not in seen:
            cleaned_parts.append(part)
            seen.add(key)

    return cleaned_parts


# Repair common text generation mistakes in the assembled output
def fix_common_output_errors(text):
    text = safe(text)
    if not text:
        return ""

    fixes = [
        (r"\bit is indicated for ([^.]*?) is available\b", r"It is indicated for \1."),
        (r"\bit works through ([^.]*?) is a\b", r"It works through \1, which is a"),
        (r"\bit works through ([^.]*?) is an\b", r"It works through \1, which is an"),
        (r"\bit works through r763 is\b", "It works through R763, which is"),
        (r"\bits mechanism of action involves the use of\b", "Its mechanism of action involves"),
        (r"\bit works through the use of\b", "It works through"),
        (r"\bit is also known as\s+\.", ""),
        (r"\bit is also known by the following synonyms:\s+\.", ""),
    ]

    for pattern, replacement in fixes:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    return text


# Validate that the generated output text is long enough and free from common error patterns
def is_valid_output(text):
    text = safe(text)
    if not text:
        return False

    if len(text.split()) < 15:
        return False

    bad_patterns = [
        r"\bit is indicated for is\b",
        r"\bit works through is\b",
        r"\bit works by is\b",
        r"\bits mechanism of action involves is\b",
        r"\bit is also known as\s*\.",
        r"\bit is also known by the following synonyms:\s*\.",
        r"\bNone\b",
        r"\bnan\b"
    ]

    for pattern in bad_patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return False

    return True


# Build the final natural language target output using one of several sentence order templates
def build_output_text(row, template_version=1):
    name = clean_field_text(safe(row.get("name")))
    groups = normalize_group_values(safe(row.get("groups")), max_groups=2)
    desc = clean_description_text(safe(row.get("description")))
    indication = clean_field_text(safe(row.get("indication")))
    moa = clean_field_text(safe(row.get("mechanism-of-action")))
    state = clean_field_text(safe(row.get("state")))
    synonyms = normalize_synonyms_text(safe(row.get("synonyms")), main_name=name, max_synonyms=2)
    unii = clean_field_text(safe(row.get("unii")))
    avg_mass = clean_field_text(safe(row.get("average-mass")))
    mono_mass = clean_field_text(safe(row.get("monoisotopic-mass")))
    cas = clean_field_text(safe(row.get("cas-number")))
    drugbank_id = clean_field_text(safe(row.get("drugbank-id")))

    opening = build_opening_sentence(name, groups, drugbank_id)
    syn_sent = build_synonym_sentence(synonyms, row.get("synonyms"))
    indication_sent = build_indication_sentence(indication)
    mechanism_sent = build_mechanism_sentence(moa)
    physical_sent = build_physical_sentence(state, avg_mass, mono_mass)
    registry_sent = build_registry_sentence(unii, cas)

    # Arrange sentence parts using alternative templates to create output variation
    if template_version == 1:
        parts = [opening, syn_sent, desc, indication_sent, mechanism_sent, physical_sent, registry_sent]
    elif template_version == 2:
        parts = [opening, desc, syn_sent, indication_sent, mechanism_sent, registry_sent, physical_sent]
    else:
        parts = [opening, syn_sent, desc, mechanism_sent, indication_sent, physical_sent, registry_sent]

    # Deduplicate, clean, and finalize the complete output paragraph
    parts = deduplicate_sentences(parts)

    final_text = " ".join(parts).strip()
    final_text = re.sub(r"\s+", " ", final_text).strip()
    final_text = fix_common_output_errors(final_text)

    return final_text


# Define multiple instruction prompt variants to diversify the KG-to-text training dataset
instruction_options = [
    "Convert the following drug knowledge graph triples into a clear and complete natural language description.",
    "Generate a fluent medical description from the following DrugBank knowledge graph triples.",
    "Write a clear and grammatically correct summary of the drug using the following knowledge graph triples.",
    "Transform the following DrugBank triples into a natural language drug description.",
    "Produce a coherent medical summary from the following structured drug knowledge graph facts.",
    "Create a complete and readable drug description from the following DrugBank triples."
]

# Create working copies of the triple dataset and cleaned tabular dataset
triples_df = triples_df_v2.copy()
clean_df = clean_12_df.copy()

# Standardize whitespace in triple and cleaned dataset key columns
triples_df["subject"] = triples_df["subject"].astype(str).str.strip()
triples_df["predicate"] = triples_df["predicate"].astype(str).str.strip()
triples_df["object"] = triples_df["object"].astype(str).str.strip()

clean_df["drugbank-id"] = clean_df["drugbank-id"].astype(str).str.strip()

# Remove duplicate triples and prepare a lookup table keyed by DrugBank ID
triples_df = triples_df.drop_duplicates(subset=["subject", "predicate", "object"]).reset_index(drop=True)
clean_df_lookup = clean_df.drop_duplicates(subset=["drugbank-id"]).set_index("drugbank-id", drop=False)

# Create an empty list to store final KG-to-text training examples
training_rows = []

# Build one training example per drug by combining grouped triples with generated target text
for subject_id, group in triples_df.groupby("subject", sort=True):
    if subject_id not in clean_df_lookup.index:
        continue

    row_data = clean_df_lookup.loc[subject_id]
    input_text = triples_to_input_text(group)

    # Skip examples with empty triple input text
    if not input_text:
        continue

    # Randomly choose one output template version for variation in generated targets
    template_version = random.choice([1, 2, 3])
    output_text = build_output_text(row_data, template_version=template_version)

    # Skip examples with empty or invalid generated target text
    if not output_text:
        continue

    if not is_valid_output(output_text):
        continue

    # Randomly choose one instruction prompt for instruction-tuning diversity
    instruction_text = random.choice(instruction_options)

    # Store the final instruction-input-output training example
    training_rows.append({
        "drugbank_id": subject_id,
        "instruction": instruction_text,
        "input": input_text,
        "output": output_text
    })

# Convert the generated training examples into a DataFrame and remove exact duplicates
kg_text_df = pd.DataFrame(training_rows)
kg_text_df = kg_text_df.drop_duplicates(
    subset=["drugbank_id", "instruction", "input", "output"]
).reset_index(drop=True)

# Print the final KG-to-text dataset size and preview sample training examples
print("KG-to-text dataset shape:", kg_text_df.shape)
display(kg_text_df.head())

**Perform KG-to-Text Pre-Split Quality Checks**


In [ ]:
print("KG-to-text dataset shape:", kg_text_df.shape)
print("Unique drugbank_id count:", kg_text_df["drugbank_id"].nunique())

print("\nNull values in kg_text_df:")
display(kg_text_df.isna().sum().to_frame("null_count"))

print("\nDuplicate checks:")
print("Duplicate full rows:", kg_text_df.duplicated().sum())
print("Duplicate drugbank_id:", kg_text_df.duplicated(subset=["drugbank_id"]).sum())
print("Duplicate input:", kg_text_df.duplicated(subset=["input"]).sum())
print("Duplicate output:", kg_text_df.duplicated(subset=["output"]).sum())
print("Duplicate instruction+input+output:", kg_text_df.duplicated(subset=["instruction", "input", "output"]).sum())

**Measure Triple Counts Per Drug**


In [ ]:
triple_count_df = (
    triples_df.groupby("subject")
    .size()
    .reset_index(name="triple_count")
    .rename(columns={"subject": "drugbank_id"})
)

print("Triple count summary:")
display(triple_count_df.head())

print("Average triples per drug:", triple_count_df["triple_count"].mean())
print("Min triples per drug:", triple_count_df["triple_count"].min())
print("Max triples per drug:", triple_count_df["triple_count"].max())

**Validate Required Predicate Coverage**


In [ ]:
# Define the list of required predicates that every drug should contain in the triple dataset
required_predicates = [
    "rdf:type",
    "hasName",
    "hasDrugBankID",
    "hasGroup",
    "hasSynonym",
    "hasState",
    "hasDescription",
    "hasIndication",
    "hasMechanismOfAction",
    "hasUNII",
    "hasCASNumber",
    "hasAverageMass",
    "hasMonoisotopicMass"
]

# Build a subject-predicate presence matrix to track which predicates exist for each drug
predicate_presence = (
    triples_df.assign(present=1)
    .pivot_table(
        index="subject",
        columns="predicate",
        values="present",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
    .rename(columns={"subject": "drugbank_id"})
)

# Add any missing required predicate columns with default absence value
for p in required_predicates:
    if p not in predicate_presence.columns:
        predicate_presence[p] = 0

# Identify which required predicates are missing for each drug
predicate_presence["missing_required_predicates"] = predicate_presence[required_predicates].apply(
    lambda row: [p for p in required_predicates if row[p] == 0],
    axis=1
)

# Count the number of missing required predicates for each drug
predicate_presence["missing_required_count"] = predicate_presence["missing_required_predicates"].apply(len)

# Display drugs that are missing one or more required predicates
print("Drugs with missing required predicates:")
display(
    predicate_presence[predicate_presence["missing_required_count"] > 0][
        ["drugbank_id", "missing_required_count", "missing_required_predicates"]
    ].head(20)
)

# Print the total number of drugs with at least one missing required predicate
print("Number of drugs with at least one missing required predicate:",
      (predicate_presence["missing_required_count"] > 0).sum())

**Check Output Text for None and NaN Leakage**


In [ ]:
import re

def find_none_nan_context(text):
    text = "" if pd.isna(text) else str(text)
    matches = re.findall(r".{0,25}\b(?:None|nan)\b.{0,25}", text, flags=re.IGNORECASE)
    return matches

kg_text_df["none_nan_context"] = kg_text_df["output"].apply(find_none_nan_context)

problem_rows = kg_text_df[kg_text_df["none_nan_context"].apply(len) > 0]

print("Rows with standalone None/nan:", len(problem_rows))
display(problem_rows[["drugbank_id", "none_nan_context", "output"]].head(10))

**Build Full KG-to-Text Audit Table**

In [ ]:
kg_audit_df = kg_text_df.merge(triple_count_df, on="drugbank_id", how="left")
kg_audit_df = kg_audit_df.merge(
    predicate_presence[["drugbank_id", "missing_required_count", "missing_required_predicates"]],
    on="drugbank_id",
    how="left"
)

kg_audit_df["input_word_count"] = kg_audit_df["input"].fillna("").apply(lambda x: len(str(x).split()))
kg_audit_df["output_word_count"] = kg_audit_df["output"].fillna("").apply(lambda x: len(str(x).split()))
kg_audit_df["instruction_word_count"] = kg_audit_df["instruction"].fillna("").apply(lambda x: len(str(x).split()))

print("Full audit dataframe shape:", kg_audit_df.shape)
display(kg_audit_df.head())

**Build KG-to-Text Summary Table**


In [ ]:
kg_summary = pd.DataFrame([
    {"metric": "total_kg_text_rows", "value": len(kg_text_df)},
    {"metric": "unique_drugbank_ids", "value": kg_text_df["drugbank_id"].nunique()},
    {"metric": "null_drugbank_id", "value": kg_text_df["drugbank_id"].isna().sum()},
    {"metric": "null_instruction", "value": kg_text_df["instruction"].isna().sum()},
    {"metric": "null_input", "value": kg_text_df["input"].isna().sum()},
    {"metric": "null_output", "value": kg_text_df["output"].isna().sum()},
    {"metric": "duplicate_full_rows", "value": kg_text_df.duplicated().sum()},
    {"metric": "duplicate_drugbank_id", "value": kg_text_df.duplicated(subset=["drugbank_id"]).sum()},
    {"metric": "avg_triples_per_drug", "value": round(triple_count_df["triple_count"].mean(), 2)},
    {"metric": "min_triples_per_drug", "value": triple_count_df["triple_count"].min()},
    {"metric": "max_triples_per_drug", "value": triple_count_df["triple_count"].max()},
    {"metric": "drugs_with_missing_required_predicates", "value": (predicate_presence["missing_required_count"] > 0).sum()},
    {"metric": "avg_input_word_count", "value": round(kg_audit_df["input_word_count"].mean(), 2)},
    {"metric": "avg_output_word_count", "value": round(kg_audit_df["output_word_count"].mean(), 2)},
])

display(kg_summary)

**Save KG-to-Text Audit Reports**


In [ ]:
kg_audit_df.to_csv(os.path.join(OUTPUT_DIR, "kg_text_pre_split_audit.csv"), index=False)
kg_summary.to_csv(os.path.join(OUTPUT_DIR, "kg_text_pre_split_summary.csv"), index=False)

print("Saved:")
print("-", os.path.join(OUTPUT_DIR, "kg_text_pre_split_audit.csv"))
print("-", os.path.join(OUTPUT_DIR, "kg_text_pre_split_summary.csv"))

**Create Train, Validation, and Test Splits**

In [ ]:
from sklearn.model_selection import train_test_split
import os
import json
import pandas as pd

#CHECK DATASET
print("KG-to-text dataset shape:", kg_text_df.shape)
print("Columns:", kg_text_df.columns.tolist())
print("Unique drugbank_id count:", kg_text_df["drugbank_id"].nunique())
print("Duplicate drugbank_id rows:", kg_text_df.duplicated(subset=["drugbank_id"]).sum())

#SPLIT BY UNIQUE DRUGBANK ID
drug_ids = kg_text_df["drugbank_id"].drop_duplicates().tolist()

train_ids, temp_ids = train_test_split(
    drug_ids,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train IDs:", len(train_ids))
print("Validation IDs:", len(val_ids))
print("Test IDs:", len(test_ids))

#CREATE SPLIT DATAFRAMES
train_df = kg_text_df[kg_text_df["drugbank_id"].isin(train_ids)].reset_index(drop=True)
val_df = kg_text_df[kg_text_df["drugbank_id"].isin(val_ids)].reset_index(drop=True)
test_df = kg_text_df[kg_text_df["drugbank_id"].isin(test_ids)].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

#LEAKAGE CHECK

print("Overlap checks:")
print("Train ∩ Validation:", len(set(train_df["drugbank_id"]) & set(val_df["drugbank_id"])))
print("Train ∩ Test:", len(set(train_df["drugbank_id"]) & set(test_df["drugbank_id"])))
print("Validation ∩ Test:", len(set(val_df["drugbank_id"]) & set(test_df["drugbank_id"])))

# SPLIT SUMMARY
split_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_df),
        "unique_drugbank_id": train_df["drugbank_id"].nunique()
    },
    {
        "split": "validation",
        "rows": len(val_df),
        "unique_drugbank_id": val_df["drugbank_id"].nunique()
    },
    {
        "split": "test",
        "rows": len(test_df),
        "unique_drugbank_id": test_df["drugbank_id"].nunique()
    }
])

print("\nSplit summary:")
display(split_summary)

#FINAL TRAINING DATAFRAMES
# KEEP ONLY instruction, input, output
train_final = train_df[["instruction", "input", "output"]].copy()
val_final = val_df[["instruction", "input", "output"]].copy()
test_final = test_df[["instruction", "input", "output"]].copy()

print("Train final shape:", train_final.shape)
print("Validation final shape:", val_final.shape)
print("Test final shape:", test_final.shape)

#SAVE CSV FILES
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_csv_path = os.path.join(OUTPUT_DIR, "kg_to_text_train.csv")
val_csv_path = os.path.join(OUTPUT_DIR, "kg_to_text_validation.csv")
test_csv_path = os.path.join(OUTPUT_DIR, "kg_to_text_test.csv")

train_final.to_csv(train_csv_path, index=False)
val_final.to_csv(val_csv_path, index=False)
test_final.to_csv(test_csv_path, index=False)

print("\nSaved CSV files:")
print("-", train_csv_path)
print("-", val_csv_path)
print("-", test_csv_path)

#SAVE JSONL FILES
def save_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "instruction": row["instruction"],
                "input": row["input"],
                "output": row["output"]
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

train_jsonl_path = os.path.join(OUTPUT_DIR, "kg_to_text_train.jsonl")
val_jsonl_path = os.path.join(OUTPUT_DIR, "kg_to_text_validation.jsonl")
test_jsonl_path = os.path.join(OUTPUT_DIR, "kg_to_text_test.jsonl")

save_jsonl(train_final, train_jsonl_path)
save_jsonl(val_final, val_jsonl_path)
save_jsonl(test_final, test_jsonl_path)

print("\nSaved JSONL files:")
print("-", train_jsonl_path)
print("-", val_jsonl_path)
print("-", test_jsonl_path)

#FINAL PREVIEW
print("\nTrain preview:")
display(train_final.head(3))

print("Validation preview:")
display(val_final.head(3))

print("Test preview:")
display(test_final.head(3))